In [1]:
import os
import pandas as pd
import pyarrow.dataset as ds
import gcsfs

In [3]:
# initialize GCS file system for reading data from GCS
fs = gcsfs.GCSFileSystem()

In [4]:
# GCS bucket path
gcs_base_path = "gs://arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/"

# Set STARsolo feature type to Velocyto
feature_type = "Velocyto"

# helper function to list files 
def get_file_table(gcs_base_path: str, target: str=None, endswith: str=None):
    files = fs.glob("/".join([gcs_base_path.rstrip("/"), "**"]))
    if target:
        files = [f for f in files if os.path.basename(f) == target]
    else:
        files = [f for f in files if f.endswith(endswith)]
    file_list = []
    for f in files:
        file_list.append(f.split("/")[-2:-1] + [f])
    return pd.DataFrame(file_list, columns=["organism", "file_path"])

In [21]:
gcs_path = "/".join([gcs_base_path.rstrip("/"), "metadata", feature_type])

# list files
sample_pq_files = get_file_table(gcs_path, "sample_metadata.parquet")
print(sample_pq_files.shape)
sample_pq_files.head()

# list files
obs_pq_files = get_file_table(gcs_path, "obs_metadata.parquet")
print(obs_pq_files.shape)
obs_pq_files.head()

(27, 2)
(27, 2)


,organism,file_path
0,Arabidopsis_thaliana,arc-institute-virtual-cell-atlas/scbasecount/2...
1,Bos_taurus,arc-institute-virtual-cell-atlas/scbasecount/2...
2,Caenorhabditis_elegans,arc-institute-virtual-cell-atlas/scbasecount/2...
3,Callithrix_jacchus,arc-institute-virtual-cell-atlas/scbasecount/2...
4,Chlorocebus_aethiops,arc-institute-virtual-cell-atlas/scbasecount/2...


In [22]:
# set the path
gcs_path = "/".join([gcs_base_path.rstrip("/"), "h5ad", feature_type])

# list files
h5ad_files = get_file_table(gcs_path, endswith=".h5ad")
print(h5ad_files.shape)
h5ad_files.head()

(61297, 2)


,organism,file_path
0,Arabidopsis_thaliana,arc-institute-virtual-cell-atlas/scbasecount/2...
1,Arabidopsis_thaliana,arc-institute-virtual-cell-atlas/scbasecount/2...
2,Arabidopsis_thaliana,arc-institute-virtual-cell-atlas/scbasecount/2...
3,Arabidopsis_thaliana,arc-institute-virtual-cell-atlas/scbasecount/2...
4,Arabidopsis_thaliana,arc-institute-virtual-cell-atlas/scbasecount/2...


In [23]:
# Display all organisms
sample_pq_files["organism"].unique()

array(['Arabidopsis_thaliana', 'Bos_taurus', 'Caenorhabditis_elegans',
       'Callithrix_jacchus', 'Chlorocebus_aethiops', 'Danio_rerio',
       'Drosophila_melanogaster', 'Equus_caballus', 'Gallus_gallus',
       'Gasterosteus_aculeatus', 'Gorilla_gorilla',
       'Heterocephalus_glaber', 'Homo_sapiens', 'Macaca_mulatta',
       'Mesocricetus_auratus', 'Monodelphis_domestica', 'Mus_musculus',
       'Oryctolagus_cuniculus', 'Oryza_sativa', 'Ovis_aries',
       'Pan_troglodytes', 'Rattus_norvegicus', 'Schistosoma_mansoni',
       'Solanum_lycopersicum', 'Sus_scrofa', 'Taeniopygia_guttata',
       'Zea_mays'], dtype=object)

In [26]:
# Look for zebrafish retina
infile = sample_pq_files[sample_pq_files["organism"] == "Danio_rerio"]["file_path"].values[0]

# load the metadata
sample_metadata = ds.dataset(infile, filesystem=fs, format="parquet").to_table().to_pandas()
print(sample_metadata.shape)
sample_metadata[sample_metadata["tissue"]=="retina"]

(842, 17)


,entrez_id,srx_accession,file_path,obs_count,lib_prep,tech_10x,cell_prep,organism,tissue,tissue_ontology_term_id,disease,disease_ontology_term_id,perturbation,cell_line,antibody_derived_tag,czi_collection_id,czi_collection_name
27,28607953,SRX21172619,gs://arc-institute-virtual-cell-atlas/scbaseco...,16064,10x_Genomics,multiome,single_nucleus,Danio rerio,retina,UBERON:0000966,unsure,None,control (injury control),not applicable,no,None,None
171,26824053,SRX19537708,gs://arc-institute-virtual-cell-atlas/scbaseco...,4185,10x_Genomics,3_prime_gex,single_cell,Danio rerio,retina,UBERON:0000966,none,None,uninjured (control),none,no,None,None
174,26824055,SRX19537710,gs://arc-institute-virtual-cell-atlas/scbaseco...,7044,10x_Genomics,3_prime_gex,single_cell,Danio rerio,retina,UBERON:0000966,not specified,None,4 days post-light lesion,Tg(pcna:EGFP); Tg(gfap:nls-mCherry),no,None,None
423,29523718,SRX21788827,gs://arc-institute-virtual-cell-atlas/scbaseco...,27622,10x_Genomics,multiome,single_nucleus,Danio rerio,retina,UBERON:0000966,unsure,None,"NMDA, 72 hours post-treatment",not_applicable,no,None,None
540,36611028,SRX27130821,gs://arc-institute-virtual-cell-atlas/scbaseco...,15674,10x_Genomics,3_prime_gex,single_cell,Danio rerio,retina,UBERON:0000966,"none (injury model, not disease)",None,optic nerve transection (1 day post-injury),isl2b:eGFP transgenic zebrafish,no,None,None
651,29523721,SRX21788830,gs://arc-institute-virtual-cell-atlas/scbaseco...,28182,10x_Genomics,multiome,single_nucleus,Danio rerio,retina,UBERON:0000966,unsure,None,NMDA (treatment at 96 hours post-injury),not_applicable,no,None,None
774,31687606,SRX23456399,gs://arc-institute-virtual-cell-atlas/scbaseco...,29896,10x_Genomics,other,single_nucleus,Danio rerio,retina,UBERON:0000966,none,None,"NMDA treatment, 36 hours post-injury",none,no,None,None
798,29523729,SRX21788838,gs://arc-institute-virtual-cell-atlas/scbaseco...,26663,10x_Genomics,multiome,single_nucleus,Danio rerio,retina,UBERON:0000966,injury-induced neurogenesis,MONDO:0021178,"light damage, 96 hours post-treatment",not_applicable,no,None,None
827,26824057,SRX19537712,gs://arc-institute-virtual-cell-atlas/scbaseco...,5486,10x_Genomics,3_prime_gex,single_cell,Danio rerio,retina,UBERON:0000966,"cancer progression, specifically sarcomas",MONDO:0005089,6 days post-light lesion,Tg(pcna:EGFP); Tg(gfap:nls-mCherry) zebrafish,no,None,None


## Dataset: Zebrafish retina treated with NMDA
**Paper:** [Common and divergent gene regulatory networks control injury...](https://www.nature.com/articles/s41586-020-2850-z) (Nature)


**GEO Accession:** [GSE239410](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE239410)

| SRX Accession | GSM Accession | Perturbation |
| :--- | :--- | :--- |
| SRX21172619 | GSM7664321 | Control (uninjured) |
| SRX23456399 | GSM8047854 | 36 hours post-NMDA injection |
| SRX21788827 | GSM7784425 | 72 hours post-NMDA injection |
| SRX21788830 | GSM7784426 | 96 hours post-NMDA injection |